## Phi-4 RAG ಜೊತೆ Azure AI ಹುಡುಕಾಟ

ಈ ನೋಟ್‌ಬುಕ್ Phi-4-mini ಮತ್ತು Phi-4-multimodal ಅನ್ನು Retrieval Augmented Generation (RAG) ಜೊತೆಗೆ Azure AI ಹುಡುಕಾಟಕ್ಕಾಗಿ ಹೇಗೆ ಬಳಸುವುದು ಎಂಬುದನ್ನು ತೋರಿಸುತ್ತದೆ. ಇದು ಏಕ-ಮಾದರಿ (ಪಠ್ಯ-ಮಾತ್ರ) ಮತ್ತು ಬಹು-ಮಾದರಿ (ಪಠ್ಯ ಮತ್ತು ಚಿತ್ರ) ಸಂದರ್ಭಗಳನ್ನು ಒಳಗೊಂಡಿದೆ.

**ಅಗತ್ಯತೆಗಳು:**
*   Azure AI Search ವೇಕ್ಟರ್ ಸೂಚ್ಯಂಕ (ಒಂದು ರಚಿಸಲು [ಈ ನಿರ್ದೇಶನಗಳನ್ನು](https://learn.microsoft.com/azure/search/search-get-started-portal-import-vectors?tabs=sample-data-storage%2Cmodel-aoai%2Cconnect-data-storage) ಅನುಸರಿಸಿ)
*   Phi-4-mini ಅಥವಾ Phi-4-multimodal ಅಂತ್ಯಬિંદುಗಳು Microsoft Foundry ನಲ್ಲಿ ಅಳವಡಿಸಲಾಗಿದೆ


In [ ]:
! pip install azure-ai-inference azure-search-documents

### ಪಠ್ಯ-ಮಾತ್ರ RAG Phi-4-mini ಜೊತೆಗೆ

ಈ ವಿಭಾಗವು Phi-4-mini ಅನ್ನು ಚಾಟ್ ಪೂರ್ಣಗೊಳಿಸುವ ಮಾದರಿಯಾಗಿ RAG ಗೆ ಬಳಸುವ ವಿಧಾನವನ್ನು ತೋರಿಸುತ್ತದೆ, naanị ಪಠ್ಯವನ್ನು ಇನ್‌ಪುಟ್ ಆಗಿ ಬಳಸಿ. ಇದರಲ್ಲಿ Microsoft Foundry Inference ಮತ್ತು Azure AI Search ಗೆ ಸಂಪರ್ಕ ಸಾಧಿಸಿ, సంబంధಿತ ದಾಖಲೆಗಳನ್ನು ಪಡೆದು, ಪಡೆದುಕೊಂಡ ಸಂದರ್ಭವನ್ನು ಉಪಯೋಗಿಸಿ ಉತ್ತರವನ್ನು ರಚಿಸುವುದು ಒಳಗೊಂಡಿದೆ.


In [ ]:
# Step 1: Connect to Microsoft Foundry Inference & Azure AI Search
import os
from azure.ai.inference import ChatCompletionsClient
from azure.ai.inference.models import UserMessage
from azure.core.credentials import AzureKeyCredential
from azure.search.documents import SearchClient
from azure.search.documents.models import VectorizableTextQuery

chat_client = ChatCompletionsClient(
    endpoint=os.environ["AZURE_INFERENCE_ENDPOINT"], # Phi-4-mini endpoint
    credential=AzureKeyCredential(os.environ["AZURE_INFERENCE_CREDENTIAL"]),
)

search_client = SearchClient(
    endpoint=os.environ["AZURE_SEARCH_ENDPOINT"],
    index_name=os.environ["AZURE_SEARCH_INDEX"],
    credential=AzureKeyCredential(os.environ["AZURE_SEARCH_KEY"])
)

# Step 2: Retrieve relevant documents from Azure AI Search
def retrieve_documents(query: str, top: int = 10):
    vector_query = VectorizableTextQuery(text=query, k_nearest_neighbors=top, fields="text_vector")
    results = search_client.search(search_text=query,vector_queries=[vector_query],select=["content"],top=top)
    return [doc["content"] for doc in results]

# Step 3: Generate answer using RAG!
def generate_rag_response(query: str):
    docs = retrieve_documents(query)
    context = "\n---\n".join(docs)
    prompt = f"""You are a helpful assistant. Use only the following context to answer the question. If the answer isn't in the context, say 'I don't know'.
    Context: {context} Question: {query} Answer:"""
    response = chat_client.complete(messages=[UserMessage(content=prompt)])
    return response.choices[0].message.content

# Example usage
user_query = "What is the capital of France?"
answer = generate_rag_response(user_query)
print(f"Q: {user_query}\nA: {answer}")


### Phi-4-multimodal ಹೊಂದಿಸಿದ ಬಹು-ಮಾದರಿ RAG

ಈ ಭಾಗವು ಪಠ್ಯ ಮತ್ತು ಚಿತ್ರ ಇನ್‌ಪುಟ್ ಇಬ್ಬರನ್ನೂ ಒಳಗೊಂಡು RAG ಗೆ ಚಾಟ್ ಪೂರ್ಣಗೊಳಿಸುವ ಮಾದರಿಯಾಗಿ Phi-4-multimodal ಅನ್ನು ಹೇಗೆ ಬಳಸಬೇಕು ಎಂಬುದನ್ನು ತೋರಿಸುತ್ತದೆ. ಇದು Azure AI ನಿರೀಕ್ಷಣೆ ಮತ್ತು Azure AI ಶೋಧನೆಗೆ ಸಂಪರ್ಕ ಸಾಧಿಸುವುದು, ಸಂಬಂಧಿಸಿದ ದಾಖಲೆಗಳನ್ನು ಪಡೆಯುವುದು, ಮತ್ತು ಬಹು-ಮಾದರಿ ಪ್ರತಿಕ್ರಿಯೆಯನ್ನು ರಚಿಸುವುದನ್ನು ಒಳಗೊಂಡಿದೆ.

**ನೋಟ್:** ನಿಮ್ಮ Azure AI ಶೋಧನೆ ಸೂಚ್ಯಂಕದಲ್ಲಿ `text_vector` ಮತ್ತು `image_vector` ಕ್ಷೇತ್ರಗಳಿದ್ದರೆ ನೀವು ಬಹು-ವೆಕ್ಟರ್ ಪ್ರಶ್ನೆಯನ್ನು ಕೂಡ ನೆರವೇರಿಸಬಹುದು.


In [ ]:
# Step 1: Connect to Azure AI Inference & Azure AI Search
from azure.ai.inference import ChatCompletionsClient
from azure.ai.inference.models import (
    SystemMessage,
    UserMessage,
    TextContentItem,
    ImageContentItem,
    ImageUrl,
    ImageDetailLevel,
)
from azure.core.credentials import AzureKeyCredential
from azure.search.documents import SearchClient
from azure.search.documents.models import VectorizableTextQuery


chat_client = ChatCompletionsClient(
    endpoint=os.environ["AZURE_INFERENCE_ENDPOINT"], #Phi-4-multimodal serverless endpoint
    credential=AzureKeyCredential(os.environ["AZURE_INFERENCE_CREDENTIAL"]),
)

search_client = SearchClient(
    endpoint=os.environ["AZURE_SEARCH_ENDPOINT"],
    index_name=os.environ["AZURE_SEARCH_INDEX"],
    credential=AzureKeyCredential(os.environ["AZURE_SEARCH_KEY"])
)

# Step 2: Retrieve relevant documents from Azure AI Search
def retrieve_documents(query: str, top: int = 3):
    vector_query = VectorizableTextQuery(text=query, k_nearest_neighbors=top, fields="text_vector")
    results = search_client.search(search_text=query, vector_queries=[vector_query], select=["content"], top=top)    
    return [doc["content"] for doc in results]

## Example for Muli-modal search if you have a text_vector AND image_vector field in your vector_index
## NOTE, image vectorization is in preview at the time of writing this code, please use azure-search-documents pypi version >11.6.0b6 
# def retrieve_documents_multimodal(query: str, image_url: str, top: int = 3):
#     text_vector_query = VectorizableTextQuery(
#         text=query,
#         k_nearest_neighbors=top,
#         fields="text_vector",
#         weight=0.5  # Adjust weight as needed
#     )
#     image_vector_query = VectorizableImageUrlQuery(
#         url=image_url,
#         k_nearest_neighbors=top,
#         fields="image_vector",
#         weight=0.5  # Adjust weight as needed
#     )

#     results = search_client.search(
#         search_text=query,  
#         vector_queries=[text_vector_query, image_vector_query],
#         select=["content"],
#         top=top
#     )
#     return [doc["content"] for doc in results]


# Step 3: Generate a multimodal RAG-based answer using retrieved text and an image input
def generate_multimodal_rag_response(query: str, image_url: str):
    # Retrieve text context from search
    docs = retrieve_documents(query)
    context = "\n---\n".join(docs)

    # Build a prompt that combines the retrieved context with the user query
    prompt = f"""You are a helpful assistant. Use only the following context to answer the question. If the answer isn't in the context, say 'I don't know'.
    Context: {context} Question: {query} Answer:"""
    # Create a chat request that includes both text and image input
    response = chat_client.complete(
        messages=[
            SystemMessage(content="You are a helpful assistant that can process both text and images."),
            UserMessage(
                content=[
                    TextContentItem(text=prompt),
                    ImageContentItem(image_url=ImageUrl(url=image_url, detail=ImageDetailLevel.HIGH)),
                ]
            ),
        ]
    )
    return response.choices[0].message.content

# Example usage
user_query = "Can you search for similar items to this shoe?"
sample_image_url = "https://images.unsplash.com/photo-1542291026-7eec264c27ff?q=80&w=1770&auto=format&fit=crop&ixlib=rb-4.0.3"
answer = generate_multimodal_rag_response(user_query, sample_image_url)
print(f"Q: {user_query}\nA: {answer}")

---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**ತಿರಸ್ಕರಣೆ**:  
ಈ ದಸ್ತಾವೇಜನ್ನು AI ಅನುವಾದ ಸೇವೆಯಾದ [Co-op Translator](https://github.com/Azure/co-op-translator) ನೊಂದಿಗೆ ಅನುವಾದಿಸಲಾಗಿದೆ. ನಾವು ನಿಖರತೆಯಿಗಾಗಿ ಶ್ರಮಿಸಿದರೂ, ಸ್ವಯಂಚಾಲಿತ ಅನುವಾದಗಳಲ್ಲಿ ತಪ್ಪುಗಳು ಅಥವಾ ಅಸುಮತಿಗಳು ಇರಬಹುದು ಎಂದು ತಿಳಿದುಕೊಳ್ಳಿ. ಮೂಲ ಭಾಷೆಯಲ್ಲಿರುವ ಪ್ರಾಥಮಿಕ ದಸ್ತಾವೇಜನ್ನು ಅಧಿಕೃತ ಮೂಲ ಎಂದು গণಿಸಬೇಕು. ಪ್ರಮುಖ ಮಾಹಿತಿಗಾಗಿ, ವೃತ್ತಿಪರ ಮಾನವ ಅನುವಾದವನ್ನು ಶಿಫಾರಸು ಮಾಡಲಾಗುತ್ತದೆ. ಈ ಅನುವಾದ ಬಳಕೆಯಿಂದ ಉಂಟಾಗುವ ಯಾವುದೇ ತಪ್ಪು ಅರ್ಥಮಾಡಿಕೊಳ್ಳುವಿಕೆ ಅಥವಾ ತಪ್ಪು ವಿವರಣೆಗಳ ಹೊಣೆಗಾರಿಕೆಯನ್ನು ನಾವು ಕೈಗೊಳ್ಳುವುದಿಲ್ಲ.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
